# Watch an Agent Do ML

**A full session, from the moment the agent connects to the moment the model is live.**

This is what using TuiML through an agent actually looks like: you type a
sentence, the agent picks tools, the tool calls scroll past in the trace, and a
trained model comes out the other end.

Every tool call below is **really executed** when this notebook runs. Each one
writes a real record to the real MCP trace log (`~/.tuiml/logs/mcp.jsonl`) using
the library's own trace writer, and the trace lines you see are rendered by the
same formatter `tuiml trace` uses — so they are byte-for-byte what you would
watch scroll past in a second terminal.

The assistant's *wording* is authored for the tutorial (there is no model API
key involved), but every number it quotes is interpolated from the live result
of the call above it. Nothing is typed in by hand.

> Want the real thing? Once you have [connected your agent](/tutorials/llm_friendly/02_mcp_server.ipynb),
> paste the **You:** lines below into Claude Desktop and watch it emit these same
> calls on its own.

---

## The connection

An MCP client launches `tuiml-mcp` as a subprocess and asks it what it can do.
That handshake is the whole setup — one command wires every detected client:

```bash
tuiml setup -y      # configure all detected clients
tuiml trace -f      # (in a second terminal) follow tool calls live
```

The cell below performs the server's side of that handshake: it reports the
tool inventory the client receives on connect.

In [1]:
import os
from tuiml.agent import MCP_AVAILABLE, get_tools_for_llm

tools = get_tools_for_llm()
print(f"tuiml-mcp ready   pid={os.getpid()}")
print(f"MCP package       {'available' if MCP_AVAILABLE else 'missing'}")
print(f"tools exposed     {len(tools)}")
print()
print("  " + "\n  ".join(t["name"] for t in tools[:6]))
print(f"  … and {len(tools) - 6} more")

[tuiml] loaded 2 user algorithm(s)


tuiml-mcp ready   pid=19283
MCP package       available
tools exposed     30

  tuiml_train
  tuiml_predict
  tuiml_evaluate
  tuiml_benchmark
  tuiml_upload_data
  tuiml_save_model
  … and 24 more


## The harness

`Session` is the whole rendering layer. Turns are buffered and painted by
`show()` at the end of each cell, so one cell renders as one block of
conversation.

`call()` is the interesting part: it writes a genuine `call` record, runs the
tool through `execute_tool` (the exact function the MCP server invokes), writes
a genuine `return` record with the measured duration, then reads those two
records back out of the log and formats them with
`tuiml.cli.trace._format_record` — the function behind `tuiml trace`.

In [2]:
import html as _html
import json
import re
import time

from IPython.display import HTML, display

from tuiml.agent import execute_tool
from tuiml.agent.mcp.server import _trace_call_start, _trace_call_end, _TRACE_PATH
from tuiml.cli.trace import _format_record

_ANSI = re.compile(r"\x1b\[[0-9;]*m")

_CSS = """
<style>
.tui-chat { font-family: ui-sans-serif, system-ui, sans-serif; max-width: 46rem;
            line-height: 1.5; margin: .35rem 0; }
.tui-row  { display: flex; gap: .6rem; margin: .55rem 0; }
.tui-who  { flex: 0 0 4.6rem; font-size: .72rem; letter-spacing: .08em;
            text-transform: uppercase; opacity: .55; padding-top: .5rem; }
.tui-bub  { flex: 1; padding: .55rem .8rem; border-radius: .55rem;
            border: 1px solid rgba(128,128,128,.28); }
.tui-user .tui-bub { background: rgba(128,128,128,.10); font-weight: 500; }
.tui-trace{ font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
            font-size: .74rem; line-height: 1.75; white-space: pre-wrap;
            word-break: break-word; margin: .35rem 0 .35rem 5.2rem;
            padding: .5rem .7rem; border-radius: .45rem;
            background: rgba(128,128,128,.10);
            border-left: 2px solid rgba(128,128,128,.45); }
.tui-call { opacity: .95; }
.tui-ok   { color: #2e7d32; }
.tui-err  { color: #c62828; }
.tui-bar  { font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
            font-size: .74rem; padding: .4rem .7rem; border-radius: .45rem;
            border: 1px dashed rgba(128,128,128,.45); opacity: .8; }
@media (prefers-color-scheme: dark) {
  .tui-ok { color: #7bd88f; } .tui-err { color: #ff8a80; }
}
</style>
"""
_css_done = False


class Session:
    """Render a chat transcript whose tool calls are really executed.

    Turns accumulate in a buffer; ``show()`` paints them as one block.
    """

    def __init__(self, title):
        self._buf = [f'<div class="tui-bar">● {_html.escape(title)}</div>']

    def _bubble(self, who, text, cls=""):
        self._buf.append(
            f'<div class="tui-row {cls}">'
            f'<div class="tui-who">{who}</div>'
            f'<div class="tui-bub">{_html.escape(text)}</div>'
            f"</div>")

    def user(self, text):
        """Buffer the human turn."""
        self._bubble("You", text, "tui-user")

    def assistant(self, text):
        """Buffer the assistant turn."""
        self._bubble("Claude", text)

    def call(self, tool, **args):
        """Execute one tool for real, trace it, buffer the trace, return the result."""
        _trace_call_start(tool, args)
        t0 = time.perf_counter()
        result = execute_tool(tool, **args)
        ms = int((time.perf_counter() - t0) * 1000)
        err = result.get("error") if isinstance(result, dict) and \
            result.get("status") == "error" else None
        _trace_call_end(tool, result, ms, error=err)

        with open(_TRACE_PATH) as fh:
            recs = [json.loads(line) for line in fh][-2:]

        lines = []
        for rec in recs:
            raw = _ANSI.sub("", _format_record(rec))
            cls = "tui-call" if rec.get("phase") == "call" else (
                "tui-err" if rec.get("error") else "tui-ok")
            lines.append(f'<span class="{cls}">{_html.escape(raw)}</span>')
        self._buf.append('<div class="tui-trace">' + "\n".join(lines) + "</div>")
        return result

    def show(self):
        """Paint everything buffered since the last call, then clear."""
        global _css_done
        head = "" if _css_done else _CSS
        _css_done = True
        display(HTML(f'{head}<div class="tui-chat">' + "".join(self._buf) + "</div>"))
        self._buf.clear()


s = Session("tuiml-mcp connected — session start")
s.show()

---

## Turn 1 — "pick something and train it"

The opening move of almost every real session: the user has data and no opinion
about algorithms yet.

In [3]:
s.user("I have the iris dataset. Pick a good classifier for it and train one.")
s.assistant("Let me see what the catalogue has for tree ensembles, then train a "
            "baseline with cross-validation.")

found = s.call("tuiml_list", category="algorithm", search="random forest")

top = [c["name"].replace("tuiml_algorithm_", "") for c in found["components"][:3]]
s.assistant(f"Catalogue has {found['total']} matches; the native one is {top[0]}. "
            f"Training it on iris with 5-fold CV.")

run = s.call("tuiml_train", algorithm="RandomForestClassifier",
             data="iris", target="class", cv=5)

acc = run["metrics"]["cv_accuracy_score_mean"]
s.assistant(f"Done — model {run['model_id']}, 5-fold CV accuracy {acc:.4f}. "
            f"Want me to check it against a few alternatives before we commit?")
s.show()

## Turn 2 — "compare, then keep the winner"

One `tuiml_benchmark` call replaces the loop-over-models script most people
write by hand.

In [4]:
s.user("Yes — compare it to naive Bayes, kNN and a decision tree. Save whichever wins.")
s.assistant("Running all four over 10 folds on iris, then saving the best one.")

bench = s.call("tuiml_benchmark",
               algorithms=["RandomForestClassifier", "NaiveBayesClassifier",
                           "KNearestNeighborsClassifier", "C45TreeClassifier"],
               data="iris", target="class", cv=10,
               metrics=["accuracy_score", "f1_score"])
s.show()

print(bench["summary"])

Benchmark: classification, 4 models x 1 datasets, seed 714609390

iris  (accuracy_score)
--------------------------------------------------
  RandomForestClassifier: 0.9533 ± 0.0632
  NaiveBayesClassifier: 0.9533 ± 0.0632
  KNearestNeighborsClassifier: 0.9600 ± 0.0562  <- best
  C45TreeClassifier: 0.9600 ± 0.0562


In [5]:
import os.path

saved = s.call("tuiml_save_model", model_id=run["model_id"],
               destination="agent_model.joblib")

s.assistant(f"Benchmark done — the table above ranks all four. Saved the random "
            f"forest to {os.path.basename(saved['destination'])}.")
s.show()

## Turn 3 — the agent gets it wrong

Real sessions are not clean runs. Here the agent guesses a class name that does
not exist. This is worth watching closely: TuiML's error responses are
*structured* — `status`, `error`, `suggestion`, `recovery_tool`,
`recovery_params` — so the agent has enough to fix itself without the user
intervening.

In [6]:
s.user("Now train that forest again but with 200 trees.")
s.assistant("Retraining with n_estimators=200.")

oops = s.call("tuiml_train", algorithm="RandomForest",   # ← wrong class name
              data="iris", target="class", cv=5,
              algorithm_params={"n_estimators": 200})
s.show()

print(json.dumps(oops, indent=2)[:420])

{
  "status": "error",
  "error": "Unknown component 'RandomForest'. Use tuiml.search_algorithms('RandomForest') to find the right name, or pass the class/instance directly.",
  "error_type": "ValueError",
  "suggestion": "Check parameter types and values. Use 'tuiml_describe' to see the algorithm's parameter schema",
  "recovery_tool": "tuiml_describe",
  "recovery_params": {
    "name": null
  }
}


In [7]:
s.assistant(f"That failed: {oops['error_type']}. The response tells me to call "
            f"{oops['recovery_tool']}, so let me look up the real name first.")

lookup = s.call("tuiml_list", category="algorithm", search="RandomForest")
real_name = lookup["components"][0]["name"].replace("tuiml_algorithm_", "")

s.assistant(f"It is {real_name}, not RandomForest. Retrying.")

retry = s.call("tuiml_train", algorithm=real_name, data="iris", target="class",
               cv=5, algorithm_params={"n_estimators": 200})

s.assistant(f"Recovered — {retry['metrics']['cv_accuracy_score_mean']:.4f} "
            f"with 200 trees. No intervention needed; the error payload carried "
            f"the fix.")
s.show()

## Turn 4 — "put it behind an API"

The pipeline is the model, so serving it is one call. Nothing is exported,
converted, or re-implemented.

In [8]:
s.user("Good. Put the 200-tree one behind a REST API so I can hit it from curl.")
s.assistant("Starting a server for it on port 8931.")

srv = s.call("tuiml_serve_model", model_id=retry["model_id"],
             port=8931, host="127.0.0.1")

s.assistant(f"Live at {srv['url']} — POST /predict with a JSON body of feature "
            f"rows. Say the word and I'll shut it down.")
s.show()

In [9]:
s.user("Thanks, shut it down.")

stopped = s.call("tuiml_stop_server", server_id=srv["server_id"])
s.assistant(f"Stopped ({stopped['status']}). The saved model is still on disk, "
            f"so you can serve it again any time.")
s.show()

In [10]:
# Cleanup: the session wrote one model file next to this notebook.
import os

if os.path.exists("agent_model.joblib"):
    os.remove("agent_model.joblib")
    print("Cleaned up: agent_model.joblib")

Cleaned up: agent_model.joblib


---

## What just happened

Four turns, eleven tool calls, no glue code — and one failure the agent fixed by
itself.

| Turn | Tool calls | What it shows |
|---|---|---|
| 1. Pick and train | `tuiml_list`, `tuiml_train` | Discovery, then a cross-validated baseline |
| 2. Compare and save | `tuiml_benchmark`, `tuiml_save_model` | Many models in one call, ranked |
| 3. Recover | `tuiml_train` ✗, `tuiml_list`, `tuiml_train` | Structured errors that carry their own fix |
| 4. Serve and stop | `tuiml_serve_model`, `tuiml_stop_server` | The pipeline *is* the model, so it serves directly |

### Watching this live

Everything above wrote to the real trace log. In your own session, open a second
terminal before you start and leave it running:

```bash
tuiml trace -f                       # follow every call as it lands
tuiml trace --tool tuiml_train       # or watch one tool
tuiml trace --no-follow -n 100       # print the tail and exit
tuiml trace --clear                  # start clean before reproducing a bug
```

### Doing it for real

1. [Connect your agent](/tutorials/llm_friendly/02_mcp_server.ipynb) — `tuiml setup -y`, about five minutes.
2. Paste the four **You:** lines above into Claude Desktop, verbatim.
3. Watch `tuiml trace -f` in a second terminal. The tool calls will be the same;
   the algorithm choices may be smarter than ours.

### Where to go next

- **[Tools an Agent Can Call](/tutorials/llm_friendly/01_llm_tools.ipynb)**: every tool schema, provider conversion, and driving the same loop from Python.
- **[High-Level API](/tutorials/ml_simplified/01_high_level_api.ipynb)**: the Python twin of every call above.
- **[Case Study: Diabetes Prediction](/tutorials/case_studies/01_diabetes_prediction.ipynb)**: this pattern on a real medical dataset, end to end.